### Gated Recurrent Units(GRU):
- Gated Recurrent Units(GRU) are a type of RNN.
- The core idea behind GRUs is to use gating mechanisms to selectively update the hidden state at each time step allowing them to remember important information while discarding irrelevant details.
- GRUs aim simplify the LSTM architecture by merging some of its components and focusing on just two main gates: the update gate and reset gate.
- The GRU consists of two main gates:
  1. Update Gate(z(t)) - This gate decides how much information from previous hidden state should be retained for the next time step.
  2. Reset Gate(r(t)) - This gate determines how much of the past hidden state should be forgetten.
- These gates allow GRU to control the flow of information in a more efficient manner compared to traditional RNNs which solely rely on hidden state.

### Equations for GRU Operations:
- The internel workings of a GRU can be described using following equations:
#### 1. Reset Gate:
- The reset gate determines how much of the previous hidden state h(t-1) should be forgetten.
  - r(t) - σ(W(r).[h(t-1), x(t)])
#### 2. Update Gate:
- The update gate decides how much information from previous hidden state should be retained for the next time step.
  - z(t) - σ(W(z). [h(t-1), x(t)])
#### 3. Candidate hidden state:
- This is the potential new hidden state calculated based on the current input and the previous hidden state.
  - h'(t) = tanh(W(h). [h(t-1), x(t)])
#### 4. Hidden state:
- The final hidden state is a weighted average of the previous hidden state h(t-1) and the candidate hidden state h'(t) based on the update gate z(t).
  - h(t) = (1-z(t).h(t-1) + z(t).h'(t)

### Implementing the Gated Recurrent Unit:
#### 1. Importing Required Libraries:

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.optimizers import Adam


C:\Users\techs\anaconda3\New folder\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


#### 2. Load Dataset:
- pd.read_csv() - Reads a CSV file into a pandas DataFrame. Here, we are assuming that the dataset has a Date column which is set as the ndex of the DataFrame.
- date_parser=True - Ensures that pandas parses the 'Data' Column as datetime.
  

In [2]:
df = pd.read_csv('data.csv', parse_dates=['Date'], index_col='Date')
print(df.head())

            Temperature
Date                   
2010-01-01    27.483571
2010-01-02    24.308678
2010-01-03    28.238443
2010-01-04    32.615149
2010-01-05    23.829233


#### 3. Preprocessing the Data:
- We will scale our data to ensure all features have equal weight and avoid any bias.
- In this example, we will use MinMaxScaler, which scales the data to a range between 0 and 1.
- Proper scaling is important because neural network tend to perform better when input features are normalized.
  

In [3]:
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(df.values)

#### 4. Preparing Data for GRU:
- create_dataset() - Prepare the dataset for timeseries forecasting. It creates sliding windows of time_step length to predict the ext time step.
- X.reshape() - Reshapes the input data to fit the expected shape for the GRU which is 3D.

In [6]:
def create_dataset(data, time_step=1):
    X, y = [], []
    for i in range(len(data) - time_step - 1):
        X.append(data[i:(i+time_step), 0])
        y.append(data[i+time_step, 0])
    return np.array(X), np.array(y)

time_step = 100
X, y = create_dataset(scaled_data, time_step)
X = X.reshape(X.shape[0], X.shape[1], 1)
    

#### 5. Building the GRU Model:
- GRU(units=50) - Adds a GRU layer with 50 units (neurons).
- return_sequences=True - Ensures that the GRU layer returns the entire sequence.
- Dense(units=1) - The output layer which predicts a single value for the next time step.
- Adam() - An adaptive optimizer commonly used in deep learning.
  

In [8]:
model = Sequential()
model.add(GRU(units=50, return_sequences=True, input_shape=(X.shape[1], 1)))
model.add(GRU(units=50))
model.add(Dense(units=1))
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')


#### 6. Training the Model:
- model.fit() - Trains the model on the prepared dataset.
- The epochs=10 specifies the number of iterations over the entire dataset.
- The batch_size=32 defines the number of samples per batch.

In [9]:
model.fit(X, y, epochs=10, batch_size=32)

Epoch 1/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 12s 39ms/step - loss: 0.0209
Epoch 2/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 0.0183
Epoch 3/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 18s 73ms/step - loss: 0.0179
Epoch 4/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - loss: 0.0178 
Epoch 5/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 100ms/step - loss: 0.0179
Epoch 6/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 102ms/step - loss: 0.0179
Epoch 7/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 101ms/step - loss: 0.0179
Epoch 8/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 101ms/step - loss: 0.0179
Epoch 9/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 99ms/step - loss: 0.0177 
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 25s 101ms/step - loss: 0.0178


#### 7. Making Predictions:


In [10]:
input_sequence = scaled_data[-time_step:].reshape(1, time_step, 1)
predicted_values = model.predict(input_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step


#### 8. Inverse Trasforming the Predctions:
- scaler.inverse_transform() - Converts the normalized predictions back to their original scale.

In [11]:
predicted_values = scaler.inverse_transform(predicted_values)
print(
    f"The predicted temperature for the next day is: {predicted_values[0][0]:.2f}°C")

The predicted temperature for the next day is: 25.26°C
